In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "claude-sonnet-4-6",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [6]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c06f5f0a-45c1-4fbc-907e-f65ce63965d7'),
              AIMessage(content=[{'id': 'toolu_019Y6fwqcBn3eUgSJZYwetYG', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'green'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cf9k4b7Le1WT9gu2brfnX', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 578, 'output_tokens': 58, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-6', 'model_provider': 'anthropic'}, id='lc_run--01a0b154-60ce-7fd1-bd04

In [7]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='bd878a06-4f2f-4bb1-b780-a3b513f5e7cf'),
              AIMessage(content="Hello! I'm doing great, thanks for asking! How are you doing today? Is there anything I can help you with? 😊", additional_kwargs={}, response_metadata={'id': 'msg_011Cf9k5YMg37L9h6fhVh1Lf', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 579, 'output_tokens': 32, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-6', 'model_provider': 'anthropic'}, id='lc_run--01a0b154-9397-7181-a449-009493f94318-0', tool_calls=[], invalid_tool_calls=[], usage_meta

## Read state

In [9]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "claude-sonnet-4-6",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c97ab674-5fbc-4b92-9a8c-beba325c0e77'),
              AIMessage(content=[{'text': 'Let me save that for you right away!', 'type': 'text'}, {'id': 'toolu_01N5tRBYnadxEyWQnp4WNZAT', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'green'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cf9k7jXh4YtqqWRai3PMm', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 624, 'output_tokens': 67, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-6', '

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c97ab674-5fbc-4b92-9a8c-beba325c0e77'),
              AIMessage(content=[{'text': 'Let me save that for you right away!', 'type': 'text'}, {'id': 'toolu_01N5tRBYnadxEyWQnp4WNZAT', 'caller': {'type': 'direct'}, 'input': {'favourite_colour': 'green'}, 'name': 'update_favourite_colour', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cf9k7jXh4YtqqWRai3PMm', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 624, 'output_tokens': 67, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-4-6', '